In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
import torch

/home/info-sec-lab/BTP/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ==============================
# Configuration
# ==============================
MODEL_NAME = "microsoft/unixcoder-base"
NUM_LABELS = 2
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
EPOCHS = 3


# ==============================
# Dataset Class
# ==============================
class CodeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


# ==============================
# Data Loading Function
# ==============================
def load_and_preprocess_data(base_path, tokenizer, folder_name):
    codes = []
    labels = []
    for label_dir in ["Label_0", "Label_1"]:
        current_path = os.path.join(base_path, folder_name, label_dir)
        if not os.path.exists(current_path):
            print(f"Warning: Directory {current_path} not found. Skipping.")
            continue

        for filename in os.listdir(current_path):
            if filename.endswith(".txt"):
                filepath = os.path.join(current_path, filename)
                with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
                    code = f.read().strip()
                    # UnixCoder expects <encoder-only> input for classification
                    code = "<encoder-only>" + code
                    codes.append(code)
                labels.append(0 if label_dir == "Label_0" else 1)

    # Tokenize (UnixCoder uses the same tokenizer interface as CodeBERT)
    encodings = tokenizer(
        codes,
        truncation=True,
        padding=True,
        max_length=512,
    )
    return CodeDataset(encodings, labels)


# ==============================
# Load Tokenizer & Model
# ==============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
from transformers import RobertaModel


base_text_path = "../Text_Files/"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# Load and preprocess training data
print("Loading training & validation data...")
train_dataset = load_and_preprocess_data(base_text_path, tokenizer, "Train")

Loading training & validation data...


In [5]:
# Training arguments
training_args = TrainingArguments(
    output_dir="../checkpoints/unixcoder_only",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="no",
    save_strategy="epoch",
    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
)

In [6]:
# Trainer 
trainer = Trainer( model=model, args=training_args, train_dataset=train_dataset, ) 
print("Training model...") 
trainer.train()

Training model...


Step,Training Loss
10,0.735800
20,0.689000
30,0.701500
40,0.689500
50,0.680400
60,0.646200
70,0.688400
80,0.642200
90,0.639500
100,0.673900


TrainOutput(global_step=2322, training_loss=0.2829388842074093, metrics={'train_runtime': 557.9263, 'train_samples_per_second': 33.284, 'train_steps_per_second': 4.162, 'total_flos': 4885972298035200.0, 'train_loss': 0.2829388842074093, 'epoch': 3.0})

In [7]:
import os
current_dir = os.getcwd()
print(current_dir)

/home/info-sec-lab/BTP/SO/experiments


In [8]:
model_path = "../checkpoints/unixcoder_only/checkpoint-2322"
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Load tokenizer from the original pre-trained model (not from checkpoint)
tokenizer = AutoTokenizer.from_pretrained("microsoft/unixcoder-base")  # or whatever base model you used

# Create trainer with the loaded model
trainer = Trainer(model=model)

## 🧪 Testing on Checkpoint 3 — `UnixCoder`

In [9]:
# Evaluate on test datasets
print("Evaluating on test datasets...")
for i in range(10):
    test_folder = f"Test_{i}"
    print(f"Loading test data for {test_folder}...")
    test_dataset = load_and_preprocess_data(base_text_path, tokenizer, test_folder)
    if len(test_dataset) > 0:
        predictions = trainer.predict(test_dataset)
        # Process predictions to get labels
        predicted_labels = predictions.predictions.argmax(axis=1)
        true_labels = test_dataset.labels

        from sklearn.metrics import accuracy_score, precision_recall_fscore_support
        accuracy = accuracy_score(true_labels, predicted_labels)
        precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predicted_labels, average='binary')

        print(f"Results for {test_folder}:")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1-Score: {f1:.4f}")
    else:
        print(f"No data found for {test_folder}. Skipping evaluation.")


Evaluating on test datasets...
Loading test data for Test_0...


Results for Test_0:
  Accuracy: 0.8313
  Precision: 0.8593
  Recall: 0.7924
  F1-Score: 0.8245
Loading test data for Test_1...


Results for Test_1:
  Accuracy: 0.8014
  Precision: 0.8069
  Recall: 0.7924
  F1-Score: 0.7996
Loading test data for Test_2...


Results for Test_2:
  Accuracy: 0.7852
  Precision: 0.7769
  Recall: 0.7924
  F1-Score: 0.7846
Loading test data for Test_3...


Results for Test_3:
  Accuracy: 0.7894
  Precision: 0.7877
  Recall: 0.7924
  F1-Score: 0.7900
Loading test data for Test_4...


...

Results for Test_5:
  Accuracy: 0.8523
  Precision: 0.9002
  Recall: 0.7924
  F1-Score: 0.8429
Loading test data for Test_6...


Results for Test_6:
  Accuracy: 0.7565
  Precision: 0.7393
  Recall: 0.7924
  F1-Score: 0.7649
Loading test data for Test_7...


Results for Test_7:
  Accuracy: 0.7026
  Precision: 0.6717
  Recall: 0.7924
  F1-Score: 0.7271
Loading test data for Test_8...


Results for Test_8:
  Accuracy: 0.7565
  Precision: 0.7393
  Recall: 0.7924
  F1-Score: 0.7649
Loading test data for Test_9...


Results for Test_9:
  Accuracy: 0.7026
  Precision: 0.6717
  Recall: 0.7924
  F1-Score: 0.7271
